# Demand Prediction 2025H1 (Distributed Spark)

This notebook reproduces the paper direction with ZoneID (no lat/lon) on the first 6 months of 2025.

Main flow:
1. Read raw trips from HDFS
2. Build 10-minute pickup demand per ZoneID
3. Fill missing time bins
4. Build time-series features (lags + rolling)
5. Train and compare Linear Regression, Random Forest, GBT
6. Save metrics/predictions back to HDFS
7. Plot sample zones

In [ ]:
import json
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionZoneID2025H1")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

spark_master = spark.sparkContext.master
print(f"Spark master: {spark_master}")

try:
    cluster_json = json.load(urllib.request.urlopen("http://master:8080/json/"))
    print("Alive workers:", cluster_json.get("aliveworkers"))
except Exception as e:
    print("Could not query Spark Master REST endpoint:", e)

In [ ]:
RAW_PATH = "/user/data/raw/*.parquet"
OUT_BASE = "/user/data/results/demand_prediction/2025H1"

START_TS = "2025-01-01 00:00:00"
END_TS = "2025-07-01 00:00:00"

ZONE_COL = "PULocationID"
BIN_COL = "pickup_bin_10m"
TARGET_COL = "pickup_demand"

raw_df = spark.read.parquet(RAW_PATH)
print("Columns in raw data:", raw_df.columns)

pickup_col_candidates = [
    "tpep_pickup_datetime",
    "lpep_pickup_datetime",
    "pickup_datetime",
]
pickup_col = next((c for c in pickup_col_candidates if c in raw_df.columns), None)

if pickup_col is None:
    raise ValueError(f"Cannot find pickup timestamp column. Tried: {pickup_col_candidates}")
if ZONE_COL not in raw_df.columns:
    raise ValueError(f"Cannot find zone column: {ZONE_COL}")

trips = (
    raw_df
    .select(
        F.to_timestamp(F.col(pickup_col)).alias("pickup_ts"),
        F.col(ZONE_COL).cast("int").alias(ZONE_COL),
    )
    .where(F.col("pickup_ts").isNotNull() & F.col(ZONE_COL).isNotNull())
    .where((F.col("pickup_ts") >= F.lit(START_TS)) & (F.col("pickup_ts") < F.lit(END_TS)))
    .where(F.col(ZONE_COL) > 0)
)

demand_10m = (
    trips
    .withColumn(BIN_COL, F.to_timestamp(F.from_unixtime(F.floor(F.unix_timestamp("pickup_ts") / 600) * 600)))
    .groupBy(ZONE_COL, BIN_COL)
    .agg(F.count(F.lit(1)).cast("double").alias(TARGET_COL))
)

zone_dim = demand_10m.select(ZONE_COL).distinct()
time_bounds = demand_10m.agg(F.min(BIN_COL).alias("min_ts"), F.max(BIN_COL).alias("max_ts")).first()

if time_bounds["min_ts"] is None or time_bounds["max_ts"] is None:
    raise ValueError("No rows found after time filtering for 2025H1.")

time_dim = (
    spark.range(1)
    .select(F.sequence(F.lit(time_bounds["min_ts"]), F.lit(time_bounds["max_ts"]), F.expr("interval 10 minutes")).alias("bins"))
    .select(F.explode(F.col("bins")).alias(BIN_COL))
)

full_grid = zone_dim.crossJoin(time_dim)
demand_ts = (
    full_grid
    .join(demand_10m, on=[ZONE_COL, BIN_COL], how="left")
    .fillna({TARGET_COL: 0.0})
    .repartition(64, ZONE_COL)
    .cache()
)

print("Rows in dense demand table:", demand_ts.count())
demand_ts.orderBy(ZONE_COL, BIN_COL).show(10, truncate=False)

In [ ]:
w = Window.partitionBy(ZONE_COL).orderBy(BIN_COL)
w_hist_6 = w.rowsBetween(-6, -1)
w_hist_18 = w.rowsBetween(-18, -1)

feat_df = (
    demand_ts
    .withColumn("hour", F.hour(BIN_COL))
    .withColumn("dow", F.dayofweek(BIN_COL))
    .withColumn("month", F.month(BIN_COL))
    .withColumn("is_weekend", F.when(F.dayofweek(BIN_COL).isin([1, 7]), F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn("lag_1", F.lag(TARGET_COL, 1).over(w))
    .withColumn("lag_2", F.lag(TARGET_COL, 2).over(w))
    .withColumn("lag_3", F.lag(TARGET_COL, 3).over(w))
    .withColumn("lag_6", F.lag(TARGET_COL, 6).over(w))
    .withColumn("lag_12", F.lag(TARGET_COL, 12).over(w))
    .withColumn("lag_144", F.lag(TARGET_COL, 144).over(w))
    .withColumn("lag_1008", F.lag(TARGET_COL, 1008).over(w))
    .withColumn("roll_mean_6", F.avg(TARGET_COL).over(w_hist_6))
    .withColumn("roll_mean_18", F.avg(TARGET_COL).over(w_hist_18))
    .withColumn("roll_std_18", F.stddev_pop(TARGET_COL).over(w_hist_18))
)

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_144", "lag_1008",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

for c in feature_cols:
    feat_df = feat_df.withColumn(c, F.col(c).cast("double"))

feat_df = feat_df.dropna(subset=feature_cols + [TARGET_COL])

w_count = Window.partitionBy(ZONE_COL)
feat_df = (
    feat_df
    .withColumn("rn", F.row_number().over(w))
    .withColumn("n_zone", F.count(F.lit(1)).over(w_count))
    .withColumn("train_cut", F.floor(F.col("n_zone") * F.lit(0.7)))
    .withColumn("split", F.when(F.col("rn") <= F.col("train_cut"), F.lit("train")).otherwise(F.lit("test")))
    .cache()
)

print("Feature rows:", feat_df.count())
feat_df.groupBy("split").count().show()

train_df = feat_df.where(F.col("split") == "train")
test_df = feat_df.where(F.col("split") == "test")

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

models = [
    ("linear_regression", LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=100, regParam=0.1, elasticNetParam=0.0)),
    ("random_forest", RandomForestRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", numTrees=180, maxDepth=12, seed=42)),
    ("gbt", GBTRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=120, maxDepth=6, stepSize=0.05, seed=42)),
]

mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None

for model_name, reg in models:
    print(f"Training: {model_name}")
    pipeline = Pipeline(stages=[assembler, reg])
    fitted = pipeline.fit(train_df)
    pred = fitted.transform(test_df).withColumn("prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction")))

    mae_val = float(mae_eval.evaluate(pred))
    rmse_val = float(rmse_eval.evaluate(pred))
    mape_val = (
        pred.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred.select(
        F.lit(model_name).alias("model"),
        ZONE_COL,
        BIN_COL,
        TARGET_COL,
        F.col("prediction"),
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)

metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
print("\nModel comparison on TEST split")
display(metrics_pdf)

best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)

In [ ]:
metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")

pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

print("Saved outputs to HDFS:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")

metrics_sdf.orderBy(F.col("MAPE").asc_nulls_last()).show(truncate=False)

In [ ]:
zone_rank = (
    demand_ts.groupBy(ZONE_COL).agg(F.sum(TARGET_COL).alias("total_pickups"))
    .orderBy(F.col("total_pickups").desc())
    .limit(3)
)
top_zones = [r[ZONE_COL] for r in zone_rank.collect()]
print("Top zones for plotting:", top_zones)

best_pred = pred_union.where(F.col("model") == best_model_name).where(F.col(ZONE_COL).isin(top_zones))

plot_df = (
    best_pred
    .where(F.col(BIN_COL) >= F.to_timestamp(F.lit("2025-06-01 00:00:00")))
    .orderBy(ZONE_COL, BIN_COL)
    .toPandas()
)

if len(plot_df) == 0:
    print("No data to plot in June 2025 for selected zones.")
else:
    fig, axes = plt.subplots(len(top_zones), 1, figsize=(16, 4 * len(top_zones)), sharex=True)
    if len(top_zones) == 1:
        axes = [axes]

    for ax, z in zip(axes, top_zones):
        tmp = plot_df[plot_df[ZONE_COL] == z]
        ax.plot(tmp[BIN_COL], tmp[TARGET_COL], label="actual", linewidth=1.0)
        ax.plot(tmp[BIN_COL], tmp["prediction"], label="prediction", linewidth=1.0)
        ax.set_title(f"Zone {z} - June 2025 ({best_model_name})")
        ax.set_ylabel("pickups / 10m")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.xlabel("time")
    plt.tight_layout()
    plt.show()

## Notes

- This notebook keeps heavy ETL + feature generation + model training in Spark DataFrames (distributed).
- Only a small slice is converted to pandas for plotting.
- Outputs are written to HDFS under `/user/data/results/demand_prediction/2025H1`.